In [25]:
import cv2
import numpy as np
import datetime
import os

In [26]:
def preprocess_image(frame):
    """Aplica CLAHE e Binarização Adaptativa com janela ampla para evitar donuts."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    equalized = clahe.apply(gray)
    
    blurred = cv2.GaussianBlur(equalized, (5, 5), 0)
    
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 71, 10)
                                   
    kernel = np.ones((3,3), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)
    
    return thresh

In [27]:
def apply_watershed(thresh_roi, original_roi):
    """
    Aplica o algoritmo Watershed para separar bolhas que possam ter sido
    pintadas juntas pelo aluno.
    """
    kernel = np.ones((3,3), np.uint8)
    sure_bg = cv2.dilate(thresh_roi, kernel, iterations=2)
    
    dist_transform = cv2.distanceTransform(thresh_roi, cv2.DIST_L2, 5)
    ret, sure_fg = cv2.threshold(dist_transform, 0.5 * dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    ret, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1
    markers[unknown == 255] = 0
    
    markers = cv2.watershed(original_roi, markers)
    original_roi[markers == -1] = [0, 0, 255]
    
    return original_roi, markers

In [28]:
def order_points(pts):
    """Ordena as coordenadas: [sup-esq, sup-dir, inf-dir, inf-esq]."""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

In [29]:
def four_point_transform(image, pts):
    """Achata e recorta a região da folha para um tamanho FIXO."""
    rect = order_points(pts)
    
    LARGURA_FIXA = 600
    ALTURA_FIXA = 800

    dst = np.array([
        [0, 0],
        [LARGURA_FIXA - 1, 0],
        [LARGURA_FIXA - 1, ALTURA_FIXA - 1],
        [0, ALTURA_FIXA - 1]], dtype="float32")

    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (LARGURA_FIXA, ALTURA_FIXA))
    
    return warped

In [30]:
def align_gabarito(frame):
    """Busca os 4 quadrados pretos nos cantos (marcas de registro) para alinhar."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                   cv2.THRESH_BINARY_INV, 11, 2)
    
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    marcadores = []
    
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.04 * peri, True)
        area = cv2.contourArea(c)
        x, y, w, h = cv2.boundingRect(c)
        
        if h == 0: continue
        proporcao = float(w) / h
        
        if len(approx) == 4 and 150 < area < 5000 and 0.8 <= proporcao <= 1.2:
            M = cv2.moments(c)
            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])
                marcadores.append([cX, cY])
                
    if len(marcadores) == 4:
        pts = np.array(marcadores, dtype="float32")
        rect = order_points(pts)
        
        LARGURA_FIXA = 600
        ALTURA_FIXA = 800
        
        dst = np.array([
            [0, 0],
            [LARGURA_FIXA - 1, 0],
            [LARGURA_FIXA - 1, ALTURA_FIXA - 1],
            [0, ALTURA_FIXA - 1]], dtype="float32")

        M = cv2.getPerspectiveTransform(rect, dst)
        gabarito_alinhado = cv2.warpPerspective(frame, M, (LARGURA_FIXA, ALTURA_FIXA))
        
        contorno_folha = rect.reshape((-1, 1, 2)).astype(np.int32)
        
        return gabarito_alinhado, True, contorno_folha
    
    return frame, False, None

In [31]:
def sort_contours_top_to_bottom(contours):
    bounding_boxes = [cv2.boundingRect(c) for c in contours]
    (contours, bounding_boxes) = zip(*sorted(zip(contours, bounding_boxes),
                                             key=lambda b: b[1][1], reverse=False))
    return list(contours)

In [32]:
def order_bubbles_left_to_right(contours):
    bounding_boxes = [cv2.boundingRect(c) for c in contours]
    (contours, bounding_boxes) = zip(*sorted(zip(contours, bounding_boxes),
                                             key=lambda b: b[1][0], reverse=False))
    return list(contours)

In [ ]:
def avaliar_respostas(thresh_img, color_img, bolhas_contours, gabarito_oficial):
    """Lê os pixels de cada bolha e compara com o gabarito oficial."""
    bolhas_contours = sort_contours_top_to_bottom(bolhas_contours)
    
    acertos = 0
    total_questoes = len(gabarito_oficial)
    
    LIMIAR_PREENCHIMENTO = 1150
    
    for q, i in enumerate(range(0, len(bolhas_contours), 5)):
        if q >= total_questoes: break
        
        linha_bolhas = bolhas_contours[i:i+5]
        linha_bolhas = order_bubbles_left_to_right(linha_bolhas)
        
        pixels_marcados = []
        for c in linha_bolhas:
            mask = np.zeros(thresh_img.shape, dtype="uint8")
            cv2.drawContours(mask, [c], -1, 255, -1)
            mask = cv2.bitwise_and(thresh_img, thresh_img, mask=mask)
            total_pixels = cv2.countNonZero(mask)
            pixels_marcados.append(total_pixels)
                        
        marcadas = [idx for idx, p in enumerate(pixels_marcados) if p > LIMIAR_PREENCHIMENTO]
        
        x_status, y_status = cv2.boundingRect(linha_bolhas[-1])[:2]
        
        resposta_certa = gabarito_oficial[q]
        indice_certo = ['A', 'B', 'C', 'D', 'E'].index(resposta_certa)
        
        if len(marcadas) == 0:
            cv2.putText(color_img, "BRANCO", (x_status + 40, y_status + 15), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            cv2.drawContours(color_img, [linha_bolhas[indice_certo]], -1, (255, 150, 0), 2)
            
        elif len(marcadas) > 1:
            # Questão Anulada (Múltiplas marcações)
            cv2.putText(color_img, "ANULADA", (x_status + 40, y_status + 15), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
            for idx in marcadas:
                cv2.drawContours(color_img, [linha_bolhas[idx]], -1, (255, 0, 255), 3)
                
        else:
            indice_marcado = marcadas[0]
            resposta_aluno = ['A', 'B', 'C', 'D', 'E'][indice_marcado]
            
            if resposta_aluno == resposta_certa:
                acertos += 1
                cv2.drawContours(color_img, [linha_bolhas[indice_marcado]], -1, (0, 255, 0), 3)
            else:
                cv2.drawContours(color_img, [linha_bolhas[indice_marcado]], -1, (0, 0, 255), 3)
                cv2.drawContours(color_img, [linha_bolhas[indice_certo]], -1, (0, 255, 0), 3)

    nota = (acertos / total_questoes) * 10
    cv2.putText(color_img, f"NOTA: {nota:.1f} / 10.0", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)
    
    return color_img, nota

In [ ]:
def main(video_input=0):
    """
    Função principal. video_input pode ser 0 (webcam) ou o caminho para um arquivo .mp4
    """
    cap = cv2.VideoCapture(video_input)
    
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0 

    GABARITO_MATRIZ = ['A', 'B', 'C', 'D', 'E', 'D', 'C', 'B', 'A', 'B']
    nome_janela = "SPV - Equipe 9"
    
    if not cap.isOpened():
        print("Erro: Não foi possível abrir o vídeo/webcam.")
        return

    print("Pressione 'q' para sair.")

    modo_foto = False
    frame_alinhado_salvo = None
    
    while True:
        ret, frame = cap.read()
        if not ret: 
            break
        
        frame_camera = frame.copy()
        frame_gravacao = frame.copy()
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
            
        if not modo_foto:
            frame_alinhado, sucesso, contorno_folha = align_gabarito(frame)
            
            if sucesso:
                cv2.drawContours(frame_camera, [contorno_folha], -1, (0, 255, 0), 4)
                cv2.putText(frame_camera, "ALVO TRAVADO! Aperte 'C' para Corrigir", (20, 40), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                
                if key == ord('c'):
                    modo_foto = True
                    frame_alinhado_salvo = frame_alinhado.copy()
            else:
                cv2.putText(frame_camera, "Alinhe o gabarito na tela...", (20, 40), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            
            cv2.imshow(nome_janela, frame_camera)
            
        else:
            thresh = preprocess_image(frame_alinhado_salvo)

            cv2.imshow("DEBUG - Visao Binaria", thresh)
            contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            bolhas_encontradas = []
            for c in contours:
                x, y, w, h = cv2.boundingRect(c)
                proporcao = w / float(h)
                area = cv2.contourArea(c)
                
                if 0.7 <= proporcao <= 1.3 and 500 < area < 2500:
                    bolhas_encontradas.append(c)
                    
            resultado_visual = frame_alinhado_salvo.copy()
            
            if len(bolhas_encontradas) == 50:
                resultado_visual, nota = avaliar_respostas(thresh, resultado_visual, bolhas_encontradas, GABARITO_MATRIZ)
                
                cv2.putText(resultado_visual, "Aperte 'R' para escanear outra prova", (20, 780), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
            else:
                cv2.putText(resultado_visual, f"ERRO: Leu {len(bolhas_encontradas)}/50 bolhas.", (20, 50), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                cv2.putText(resultado_visual, "Aperte 'R' e tente tirar a foto novamente.", (20, 80), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
            cv2.imshow("Scanner Corretor", resultado_visual)
            
            cv2.putText(frame_camera, "CORRECAO CONCLUIDA NA OUTRA JANELA", (20, 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
            cv2.imshow(nome_janela, frame_camera)
            
            
            if key == ord('r'):
                modo_foto = False
                cv2.destroyWindow("Scanner Corretor")

    cap.release()
    cv2.destroyAllWindows()

In [35]:
def gerar_gabarito(filename="gabarito_teste.png"):
    largura, altura = 800, 1100
    img = np.ones((altura, largura, 3), dtype=np.uint8) * 255
    margem, tamanho_quadrado = 50, 40

    cv2.rectangle(img, (margem, margem), (margem+tamanho_quadrado, margem+tamanho_quadrado), (0,0,0), -1)
    cv2.rectangle(img, (largura-margem-tamanho_quadrado, margem), (largura-margem, margem+tamanho_quadrado), (0,0,0), -1)
    cv2.rectangle(img, (margem, altura-margem-tamanho_quadrado), (margem+tamanho_quadrado, altura-margem), (0,0,0), -1)
    cv2.rectangle(img, (largura-margem-tamanho_quadrado, altura-margem-tamanho_quadrado), (largura-margem, altura-margem), (0,0,0), -1)

    cv2.putText(img, "Gabarito - Sistema de Processamento Visual", (100, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)

    inicio_y, inicio_x, espacamento_y, espacamento_x, raio = 250, 180, 65, 80, 22
    opcoes = ['A', 'B', 'C', 'D', 'E']

    for q in range(1, 11):
        y = inicio_y + (q-1) * espacamento_y
        cv2.putText(img, f"{q:02d}.", (inicio_x - 80, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,0), 2)
        for i, letra in enumerate(opcoes):
            x = inicio_x + i * espacamento_x
            cv2.circle(img, (x, y), raio, (0,0,0), 2)
            cv2.putText(img, letra, (x - 10, y + 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,0), 2)

    cv2.imwrite(filename, img)
    print(f"Imagem de teste salva como {filename}")

In [36]:
if __name__ == "__main__":
    # gerar_gabarito()
    
    main(0)

Pressione 'q' para sair.
